# 1.2 記住資料等於學會嗎？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：先分整份文件，再切小窗口**

近似改寫的同一家族也放在同一側。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/split.svg")))

**先想一想：**同一句話重複三次，算三份新的驗證知識嗎？

先把整份文件分開，才從各自文件取窗口。若先切窗口，相鄰窗口很可能把同一句話送到兩邊，驗證就變成考背誦。

**把直覺寫成數學：**training 用來改參數；validation 用來選設定；test 留到最後。三者不是三種模型。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import split_documents

docs = ["貓看狗。", "狗看貓。", "鳥看魚。", "魚看鳥。", "貓看狗。"]
parts = split_documents(docs, seed=42)
print(parts)
assert not (set(parts["train"]) & set(parts["validation"]))

**如何讀結果：**先去完全重複，文件互不重疊；真實資料還需把改寫版本放在同一家族。

**只改一件事：**只加一份重複文件，觀察去重後的文件總數。
